In [ ]:

import numpy as np


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.0 MB/s eta 0:00:00


In [ ]:
import torch
torch.cuda.is_available()

True

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import io
import os
import random
from pathlib import Path

import requests
from PIL import Image

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.models import EfficientNet_B0_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

print(torch.__version__)


2.11.0+cu128


In [ ]:
train_dir = Path(r"/content/drive/MyDrive/Car_Demage_Severity/training")
test_dir = Path(r"/content/drive/MyDrive/Car_Demage_Severity/validation")
save_dir = Path("/content/drive/MyDrive/yolov8_train_car/pkl")
save_dir.mkdir(parents=True, exist_ok=True)

model_path = save_dir / 'cnn_car.pkl'
seed=42
random.seed(seed)
torch.manual_seed(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


device(type='cuda')

In [ ]:
import torch.nn as nn
from torchvision import transforms
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

# 1. Khởi tạo mô hình ConvNeXt_Tiny và lấy transform chuẩn của nó
weights = ConvNeXt_Tiny_Weights.DEFAULT
model = convnext_tiny(weights=weights)

# Tự động lấy mean và std chuẩn từ pre-trained weights của ConvNeXt
base_eval_transform = weights.transforms()
mean = base_eval_transform.mean
std = base_eval_transform.std

# 2. Định nghĩa kích thước ảnh đồng bộ cho cả Train và Eval (Chọn 288 hoặc 224)
# Giữ 288 để mô hình nhận diện tốt các chi tiết hư hại nhỏ của vỏ xe
image_size = 288
resize_size = 320

# 3. Cấu hình lớp Classifier Head cho ConvNeXt_Tiny (nằm ở classifier[2])
num_ftrs = model.classifier[2].in_features
# Thay len(class_names) bằng số nhãn cứng (ở đây là 3)
model.classifier[2] = nn.Linear(num_ftrs, 3)
model = model.to(device)

# 4. Cấu hình Eval Transform (Dùng cho Tập Validation và Tập Test)
eval_transform = transforms.Compose([
    transforms.Resize((resize_size, resize_size)),
    transforms.CenterCrop((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

# 5. Cấu hình Train Transform (Dùng cho Tập Train)
train_transform = transforms.Compose([
    transforms.Resize((resize_size, resize_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.12,
            contrast=0.12,
            saturation=0.08,
            hue=0.03,
        )
    ], p=0.5),
    transforms.RandomAffine(
        degrees=5,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05),
    ),
    transforms.CenterCrop((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

In [ ]:
full_train_for_train = datasets.ImageFolder(root=str(train_dir), transform=train_transform)
full_train_for_eval = datasets.ImageFolder(root=str(train_dir), transform=eval_transform)
test_dataset = datasets.ImageFolder(root=str(test_dir), transform=eval_transform)

if full_train_for_train.class_to_idx != test_dataset.class_to_idx:
    print("C?nh b?o: th? t? class train v? test kh?ng kh?p.")
    print("Train:", full_train_for_train.class_to_idx)
    print("Test :", test_dataset.class_to_idx)

targets = np.array(full_train_for_train.targets)
indices = np.arange(len(targets))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.25,
    random_state=seed,
    stratify=targets,
)

train_dataset = Subset(full_train_for_train, train_idx)
val_dataset = Subset(full_train_for_eval, val_idx)

class_names = full_train_for_train.classes
train_counts = np.bincount(targets[train_idx], minlength=len(class_names))
val_counts = np.bincount(targets[val_idx], minlength=len(class_names))

print("Classes:", class_names)
print(f"Train/Val/Test: {len(train_dataset)}/{len(val_dataset)}/{len(test_dataset)}")
print("Train class counts:", dict(zip(class_names, train_counts)))
print("Val class counts:", dict(zip(class_names, val_counts)))


Classes: ['01-minor', '02-moderate', '03-severe']
Train/Val/Test: 2209/737/390
Train class counts: {'01-minor': np.int64(807), '02-moderate': np.int64(693), '03-severe': np.int64(709)}
Val class counts: {'01-minor': np.int64(269), '02-moderate': np.int64(231), '03-severe': np.int64(237)}


In [ ]:
# DataLoader
# image_size=320 t?n VRAM h?n 224, n?n batch 16 th??ng an to?n h?n tr?n Colab/T4.
batch_size = 16
num_workers = 0
pin_memory = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

print("?? kh?i t?o c?c DataLoader th?nh c?ng!")


?? kh?i t?o c?c DataLoader th?nh c?ng!


In [ ]:
train_targets = targets[train_idx]
class_counts = np.bincount(train_targets, minlength=len(class_names))

# D?ng tr?ng s? m?m ?? tr?nh l?p ?t m?u b? ??y qu? m?nh.
raw_class_weights = class_counts.sum() / (len(class_counts) * np.maximum(class_counts, 1))
soft_class_weights = np.sqrt(raw_class_weights)
soft_class_weights = soft_class_weights / soft_class_weights.mean()
class_weights = torch.tensor(soft_class_weights, dtype=torch.float32).to(device)


class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=1.5, label_smoothing=0.03):
        super().__init__()
        self.weight = weight
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce = nn.functional.cross_entropy(
            logits,
            targets,
            weight=self.weight,
            label_smoothing=self.label_smoothing,
            reduction="none",
        )
        pt = torch.exp(-ce)
        loss = ((1.0 - pt) ** self.gamma) * ce
        return loss.mean()


criterion = FocalLoss(
    weight=class_weights,
    gamma=1.5,
    label_smoothing=0.03,
)

print("Class counts:", dict(zip(class_names, class_counts)))
print("Raw class weights:", raw_class_weights)
print("Soft class weights:", class_weights)


Class counts: {'01-minor': np.int64(807), '02-moderate': np.int64(693), '03-severe': np.int64(709)}
Raw class weights: [0.91243288 1.06253006 1.03855195]
Soft class weights: tensor([0.9536, 1.0290, 1.0174], device='cuda:0')


In [ ]:
@torch.no_grad()
def evaluate(model, loader, device, criterion=None):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    all_true = []
    all_pred = []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)

        if criterion is not None:
            loss = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)

        preds = logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

        all_true.extend(labels.cpu().numpy())
        all_pred.extend(preds.cpu().numpy())

    return {
        "loss": total_loss / max(total_samples, 1),
        "accuracy": total_correct / max(total_samples, 1),
        "macro_f1": f1_score(all_true, all_pred, average="macro", zero_division=0),
        "y_true": all_true,
        "y_pred": all_pred,
    }


def train_one_phase(
    model,
    train_loader,
    val_loader,
    device,
    epochs,
    optimizer,
    save_path,
    criterion,
    scheduler=None,
    save_metric="macro_f1",
    start_best=-1.0,
    patience=8,
    min_delta=1e-4,
    phase_name="",
):
    history = []
    best_score = start_best
    wait = 0

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        running_correct = 0
        total_samples = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            running_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_samples += labels.size(0)

        if scheduler is not None:
            scheduler.step()

        train_metrics = {
            "loss": running_loss / max(total_samples, 1),
            "accuracy": running_correct / max(total_samples, 1),
        }
        val_metrics = evaluate(model, val_loader, device, criterion)
        current_score = val_metrics[save_metric]

        if current_score > best_score + min_delta:
            best_score = current_score
            wait = 0
            torch.save(model.state_dict(), save_path)
        else:
            wait += 1

        history.append({
            "epoch": epoch,
            "train": train_metrics,
            "val": {
                "loss": val_metrics["loss"],
                "accuracy": val_metrics["accuracy"],
                "macro_f1": val_metrics["macro_f1"],
            },
        })

        prefix = f"{phase_name} " if phase_name else ""
        print(
            f"{prefix}Epoch {epoch:02d} | "
            f"train_loss={train_metrics['loss']:.4f} train_acc={train_metrics['accuracy']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['accuracy']:.4f} "
            f"val_macro_f1={val_metrics['macro_f1']:.4f}"
        )

        if wait >= patience:
            print(f"Early stopping t?i epoch {epoch} v? {save_metric} kh?ng c?i thi?n sau {patience} epoch.")
            break

    print(f"Best {save_metric}: {best_score:.4f}")
    return history, best_score


In [ ]:
import torch.nn as nn
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

# 1. Khởi tạo mô hình ConvNeXt_Tiny chính xác với weights tương ứng
weights = ConvNeXt_Tiny_Weights.DEFAULT
model = convnext_tiny(weights=weights).to(device)

# 2. Lấy số features đầu vào của classifier ConvNeXt (nằm ở mục số 2 trong classifier)
num_ftrs = model.classifier[2].in_features

# 3. Định nghĩa lại lớp tuyến tính cuối cùng phù hợp với 3 nhãn đầu ra
model.classifier[2] = nn.Linear(num_ftrs, len(class_names))
model = model.to(device)

# --- Các bước cấu hình Train Phase 1 giữ nguyên theo logic của bạn ---

# Freeze backbone, chỉ train đầu classifier head
for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

optimizer_phase1 = torch.optim.AdamW(
    model.classifier.parameters(),
    lr=3e-4,
    weight_decay=1e-4,
)

scheduler_phase1 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_phase1,
    T_max=25,
)

print("Phase 1: train ConvNeXt_Tiny classifier head")
history_phase1, best_score = train_one_phase(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=25,
    optimizer=optimizer_phase1,
    save_path=model_path,
    criterion=criterion,
    scheduler=scheduler_phase1,
    save_metric="macro_f1",
    start_best=-1.0,
    patience=7,
    phase_name="Phase 1",
)

Phase 1: train ConvNeXt_Tiny classifier head
Phase 1 Epoch 01 | train_loss=0.3945 train_acc=0.6519 | val_loss=0.3077 val_acc=0.7327 val_macro_f1=0.7274
Phase 1 Epoch 02 | train_loss=0.2693 train_acc=0.7809 | val_loss=0.2672 val_acc=0.7748 val_macro_f1=0.7709
Phase 1 Epoch 03 | train_loss=0.2403 train_acc=0.8031 | val_loss=0.2484 val_acc=0.7870 val_macro_f1=0.7844
Phase 1 Epoch 04 | train_loss=0.2213 train_acc=0.8189 | val_loss=0.2387 val_acc=0.8019 val_macro_f1=0.7976
Phase 1 Epoch 05 | train_loss=0.2112 train_acc=0.8280 | val_loss=0.2303 val_acc=0.8033 val_macro_f1=0.8028
Phase 1 Epoch 06 | train_loss=0.2038 train_acc=0.8370 | val_loss=0.2301 val_acc=0.7978 val_macro_f1=0.7981
Phase 1 Epoch 07 | train_loss=0.1963 train_acc=0.8402 | val_loss=0.2285 val_acc=0.8033 val_macro_f1=0.8036
Phase 1 Epoch 08 | train_loss=0.1942 train_acc=0.8370 | val_loss=0.2237 val_acc=0.8277 val_macro_f1=0.8231
Phase 1 Epoch 09 | train_loss=0.1926 train_acc=0.8429 | val_loss=0.2220 val_acc=0.8141 val_macro_f1

In [ ]:
import torch
import torch.nn as nn

# Phase 2: fine-tune các block cuối của ConvNeXt_Tiny + classifier
# Luôn bắt đầu phase 2 từ checkpoint tốt nhất của phase 1.
model.load_state_dict(torch.load(model_path, map_location=device))

# 1. Đóng băng toàn bộ mô hình trước
for param in model.parameters():
    param.requires_grad = False

# 2. Mở băng classifier head
for param in model.classifier.parameters():
    param.requires_grad = True

# 3. Mở băng các block cuối của ConvNeXt_Tiny (Stage 6 và Stage 7 ở cuối phần features)
for param in model.features[-2:].parameters():
    param.requires_grad = True

# 4. Cấu hình Optimizer với LR tối ưu riêng cho từng phần của ConvNeXt
optimizer_phase2 = torch.optim.AdamW([
    {"params": model.features[-2:].parameters(), "lr": 1e-5},      # LR thấp cho các block cuối
    {"params": model.classifier.parameters(), "lr": 5e-5}          # LR cao hơn một chút cho classifier head
], weight_decay=1e-4)

scheduler_phase2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_phase2,
    T_max=35,
)

print("Phase 2: fine-tune ConvNeXt_Tiny last blocks + classifier")
history_phase2, best_score = train_one_phase(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=35,
    optimizer=optimizer_phase2,
    save_path=model_path,
    criterion=criterion,
    scheduler=scheduler_phase2,
    save_metric="macro_f1",
    start_best=best_score,
    patience=10,
    phase_name="Phase 2",
)

# --- Đoạn evaluate tập Test giữ nguyên logic của bạn ---
print("\n--- Evaluating on Test Dataset ---")
model.load_state_dict(torch.load(model_path, map_location=device))
test_metrics = evaluate(model, test_loader, device, criterion)

print("Test accuracy:", test_metrics["accuracy"])
print("Test macro_f1:", test_metrics["macro_f1"])

print(classification_report(
    test_metrics["y_true"],
    test_metrics["y_pred"],
    target_names=class_names,
    zero_division=0,
))

print(confusion_matrix(
    test_metrics["y_true"],
    test_metrics["y_pred"],
))

Phase 2: fine-tune ConvNeXt_Tiny last blocks + classifier
Phase 2 Epoch 01 | train_loss=0.1735 train_acc=0.8583 | val_loss=0.2125 val_acc=0.8331 val_macro_f1=0.8326
Phase 2 Epoch 02 | train_loss=0.1604 train_acc=0.8732 | val_loss=0.2125 val_acc=0.8277 val_macro_f1=0.8274
Phase 2 Epoch 03 | train_loss=0.1437 train_acc=0.8850 | val_loss=0.2149 val_acc=0.8195 val_macro_f1=0.8195
Phase 2 Epoch 04 | train_loss=0.1272 train_acc=0.8959 | val_loss=0.2126 val_acc=0.8195 val_macro_f1=0.8184
Phase 2 Epoch 05 | train_loss=0.1189 train_acc=0.9036 | val_loss=0.2151 val_acc=0.8236 val_macro_f1=0.8228
Phase 2 Epoch 06 | train_loss=0.1055 train_acc=0.9176 | val_loss=0.2152 val_acc=0.8236 val_macro_f1=0.8247
Phase 2 Epoch 07 | train_loss=0.0932 train_acc=0.9303 | val_loss=0.2174 val_acc=0.8155 val_macro_f1=0.8173
Phase 2 Epoch 08 | train_loss=0.0855 train_acc=0.9402 | val_loss=0.2174 val_acc=0.8263 val_macro_f1=0.8264
Phase 2 Epoch 09 | train_loss=0.0801 train_acc=0.9407 | val_loss=0.2185 val_acc=0.8250

In [ ]:

def predict_from_url(url: str):
    resp = requests.get(url, timeout=20)
    resp.raise_for_status()

    image = Image.open(io.BytesIO(resp.content)).convert("RGB")
    x = eval_transform(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0].cpu()

    idx = int(torch.argmax(probs).item())
    return {
        "label": class_names[idx],
        "confidence": float(probs[idx].item()),
        "probs": {class_names[i]: float(probs[i].item()) for i in range(len(class_names))},
    }


In [ ]:

sample_url = ""

if sample_url:
    result = predict_from_url(sample_url)
    print(result)
else:
    print("H?y g?n sample_url b?ng URL ?nh xe h? h?ng ?? test predict_from_url().")


H?y g?n sample_url b?ng URL ?nh xe h? h?ng ?? test predict_from_url().
